# e1 - setup do pacote

Carrega o pacote (`pacote-e1-2026-09-13`), mostra o padrão certo de comparação entre plataformas
(posição dentro da plataforma, nunca média com média) e um gráfico de exemplo. A análise é de vocês:
as sete perguntas sugeridas estão no `README.md` do pacote, e no fim deste caderno tem o mapa
pergunta -> tabela -> gráfico.


## 1. onde está o pacote

No Colab: monta o Drive e aponta `PACOTE` pra pasta descompactada. Rodando local, aponta pra `datalake/pacote/...`.


In [ ]:
import os
import sys

NO_COLAB = "google.colab" in sys.modules
if NO_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PACOTE = "/content/drive/MyDrive/eps7008/eps7008-e1/pacote-e1-2026-09-13"  # ajuste pro caminho da pasta compartilhada
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"], check=True)
else:
    PACOTE = "../datalake/pacote/pacote-e1-2026-09-13"

assert os.path.exists(f"{PACOTE}/VERSAO.txt"), "pacote não encontrado: ajuste PACOTE"
print(open(f"{PACOTE}/VERSAO.txt").read())


In [ ]:
import duckdb
import matplotlib.pyplot as plt
import pandas as pd


## 2. as tabelas como views do duckdb

O duckdb consulta o parquet direto, sem carregar nada. `con.sql("...").df()` devolve um dataframe quando precisar do pandas.


In [ ]:
con = duckdb.connect()
for camada in ("silver", "gold"):
    for arquivo in sorted(os.listdir(f"{PACOTE}/{camada}")):
        nome = arquivo.removesuffix(".parquet")
        con.sql(f"create or replace view {nome} as select * from '{PACOTE}/{camada}/{arquivo}'")
con.sql("select view_name from duckdb_views() where not internal order by 1").df()


In [ ]:
con.sql("select * from dim_titulo limit 5").df()


In [ ]:
con.sql("""
select plataforma, count(*) as titulos, round(avg(nota), 2) as media_bruta,
  min(escala_min) as escala_de, max(escala_max) as escala_ate, median(n_votos) as votos_mediana
from fato_notas group by 1 order by 1
""").df()


## 3. o padrão: posição dentro da plataforma

`gold/notas_normalizadas` já traz `nota_z` e `nota_pct`, calculados sobre as linhas com `n_votos >= 30`.
Se precisar recalcular em outro recorte (só um país, só uma década), a regra é esta e nada mais:


In [ ]:
def normalizar(df, grupo="plataforma", nota="nota"):
    """z-score e percentil da nota dentro de cada grupo; é o que gold/notas_normalizadas faz em sql"""
    g = df.groupby(grupo)[nota]
    return df.assign(
        nota_z=(df[nota] - g.transform("mean")) / g.transform("std"),
        nota_pct=g.rank(pct=True),
    )


fato = con.sql("select * from fato_notas where n_votos >= 30").df()
normalizado = normalizar(fato)

# confere que bate com o gold
gold = con.sql("select tconst, plataforma, nota_z as nota_z_gold from notas_normalizadas").df()
conferencia = normalizado.merge(gold, on=["tconst", "plataforma"])
print("maior diferença pro gold:", (conferencia.nota_z - conferencia.nota_z_gold).abs().max())


## 4. um gráfico: por que a média engana

Esquerda: média bruta por plataforma e década. Escalas diferentes (1-10, 0-10, 0,5-5), não diz nada.
Direita: média de `nota_z` por plataforma e década, **só no quartil superior de `pct_votos_decada`** - 
o cânone de cada década, onde a cobertura do Letterboxd é plana e as populações são comparáveis.
O controle pelo quartil é o que separa "o Letterboxd gosta de filme antigo" de "o frame dos anos 1930 é só cânone".


In [ ]:
serie = con.sql("""
select d.decada, n.plataforma, avg(n.nota) as media_bruta, avg(n.nota_z) as media_z
from notas_normalizadas n
join dim_titulo d using (tconst)
where d.pct_votos_decada >= 0.75
group by 1, 2
order by 1, 2
""").df()

CORES = {"imdb": "#2a78d6", "letterboxd": "#eb6834", "ml32": "#1baf7a", "tmdb": "#eda100"}
CINZA = "#52514e"

fig, eixos = plt.subplots(1, 2, figsize=(12, 4.5), sharex=True)
paineis = (
    (eixos[0], "media_bruta", "média bruta por plataforma: escalas diferentes, não compare"),
    (eixos[1], "media_z", "média de nota_z no quartil superior de cada década"),
)
for eixo, coluna, titulo in paineis:
    for plataforma, cor in CORES.items():
        s = serie[serie.plataforma == plataforma]
        eixo.plot(s.decada, s[coluna], color=cor, linewidth=2, marker="o", markersize=4, label=plataforma)
        if coluna == "media_bruta":  # rótulo direto só onde as linhas não se cruzam na ponta
            eixo.annotate(plataforma, (s.decada.iloc[-1], s[coluna].iloc[-1]), xytext=(6, 0),
                          textcoords="offset points", va="center", fontsize=9, color=CINZA)
    eixo.set_title(titulo, fontsize=11, loc="left")
    eixo.spines[["top", "right"]].set_visible(False)
    eixo.grid(axis="y", color="#e5e4e0", linewidth=0.8)
    eixo.set_xlabel("década")
    eixo.legend(frameon=False, fontsize=9)
eixos[0].set_ylabel("nota média na escala da plataforma")
eixos[1].set_ylabel("nota_z média")
eixos[1].axhline(0, color="#c3c2b7", linewidth=1)
plt.tight_layout()


## 5. pergunta -> tabela -> gráfico sugerido

| pergunta (README) | tabela | gráfico que responde |
|---|---|---|
| o `gap_z` IMDb x Letterboxd cresce com a década, dentro de quartil? | `divergencia_par` + `dim_titulo` | linhas: `gap_z` médio por década, uma linha por quartil de `pct_votos_decada` |
| qual par mais discorda, e onde? | `divergencia_par` | barras: desvio-padrão do `gap_z` por par; depois facetar por década ou país |
| país único não-EUA vs EUA, mesma década e quartil | `divergencia_par` + `dim_titulo.pais_unico` | pontos com intervalo: `gap_z` médio por país, controlando década e quartil |
| divergência de visibilidade tem perfil? | `fato_notas` (razão de `n_votos` entre plataformas) | dispersão log-log de `n_votos` IMDb x Letterboxd, colorido por década |
| deriva: `desvio_do_ano` vs distância do lançamento | `deriva_ml32` + `dim_titulo.ano` | linhas: desvio médio por anos desde o lançamento, cânone vs não-cânone |
| gradiente de idade reaparece em algum gênero ou década? | `notas_por_faixa_etaria` + `dim_titulo` | pequenos múltiplos: `desvio_no_filme` por faixa, um painel por gênero |
| `classificacao_br` explica `gap_z`, controlando década? | `divergencia_par` + `tmdb_titulo.classificacao_br` | barras agrupadas: `gap_z` médio por classificação, uma cor por década |

Regras que valem pra todos: legenda sempre que houver mais de uma série, um eixo y por gráfico (nunca dois),
célula com menos de 30 filmes marcada e não interpretada.


## fontes e atribuição

- **IMDb** - [datasets.imdbws.com](https://datasets.imdbws.com/), uso não comercial, coletado em 13/09/2026.
- **TMDB** - *This product uses the TMDB API but is not endorsed or certified by TMDB.* Coletado em 13/09/2026.
- **MovieLens (ml-1m e ml-32m)** - F. Maxwell Harper and Joseph A. Konstan. 2015. The MovieLens Datasets: History and Context. ACM TiiS 5, 4, Article 19. [grouplens.org/datasets/movielens](https://grouplens.org/datasets/movielens/).
- **Letterboxd** - dataset *Letterboxd film ratings* de freeth no [Kaggle](https://www.kaggle.com/datasets/freeth/letterboxd-film-ratings), amostra de 11.061 usuários, dump de 10/10/2023.
- **Wikidata** - propriedade P6127 (Letterboxd film ID), CC0, extraído em 13/09/2026.
